# Notebook 05: DDIM vs DDPM 采样对比

**目标**：用同一个训好的 2D 模型，对比 DDIM 与 DDPM 采样的轨迹与结果。

**关键问题**：
1. DDIM 跳步后质量损失多大？
2. 同 x_T 下 DDIM（确定性）vs DDPM（随机）的轨迹差异？
3. DDIM `eta` 插值的影响？

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 训练数据
x_train, _ = make_moons(5000, noise=0.05)
x_train = torch.tensor(x_train, dtype=torch.float32).to(device)

T = 1000
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1 - betas
ac = alphas.cumprod(0)


In [ ]:
# 简单 MLP（同 nb04）
class TimeEmb(nn.Module):
    def __init__(self, dim=64):
        super().__init__(); self.dim = dim
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-np.log(10000) * torch.arange(half, device=t.device) / half)
        a = t[:, None].float() * freqs[None]
        return torch.cat([a.sin(), a.cos()], dim=-1)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.t = TimeEmb(64)
        self.net = nn.Sequential(
            nn.Linear(66, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 128), nn.SiLU(),
            nn.Linear(128, 2),
        )
    def forward(self, x, t):
        return self.net(torch.cat([x, self.t(t)], dim=-1))

model = MLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(2000):
    idx = torch.randint(0, len(x_train), (512,))
    x0 = x_train[idx]
    t = torch.randint(0, T, (512,), device=device)
    eps = torch.randn_like(x0)
    xt = ac[t].sqrt().unsqueeze(-1) * x0 + (1-ac[t]).sqrt().unsqueeze(-1) * eps
    F.mse_loss(model(xt, t), eps).backward()
    opt.step(); opt.zero_grad()
print('trained')

In [ ]:
@torch.no_grad()
def ddpm_sample(x_T, n_steps=T):
    """Standard DDPM ancestral sampling, 用完整 T 步"""
    model.eval()
    x = x_T.clone()
    for t in reversed(range(T)):
        t_tensor = torch.full((x.shape[0],), t, device=device, dtype=torch.long)
        eps = model(x, t_tensor)
        mean = (x - betas[t] / (1-ac[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            var = betas[t] * (1 - ac[t-1]) / (1 - ac[t])
            x = mean + var.sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x

@torch.no_grad()
def ddim_sample(x_T, n_steps=50, eta=0.0):
    """DDIM with skipping"""
    model.eval()
    x = x_T.clone()
    step = T // n_steps
    timesteps = list(range(0, T, step))[::-1]
    for i, t in enumerate(timesteps):
        t_prev = timesteps[i+1] if i+1 < len(timesteps) else -1
        t_tensor = torch.full((x.shape[0],), t, device=device, dtype=torch.long)
        eps = model(x, t_tensor)
        x0_hat = (x - (1-ac[t]).sqrt() * eps) / ac[t].sqrt()
        x0_hat = x0_hat.clamp(-3, 3)
        if t_prev < 0:
            x = x0_hat
        else:
            ac_prev = ac[t_prev]
            sigma = eta * ((1 - ac_prev)/(1-ac[t]) * (1 - ac[t]/ac_prev)).sqrt()
            dir_xt = (1 - ac_prev - sigma**2).clamp(min=0).sqrt() * eps
            noise = torch.randn_like(x) if eta > 0 else 0
            x = ac_prev.sqrt() * x0_hat + dir_xt + sigma * noise
    return x

## 实验 1：同一 x_T，DDPM vs DDIM 终态对比

In [ ]:
torch.manual_seed(123)
x_T = torch.randn(2000, 2, device=device)

# DDIM 跳步对比
configs = [('DDPM (1000 steps)', lambda: ddpm_sample(x_T)),
           ('DDIM 250 steps', lambda: ddim_sample(x_T, 250)),
           ('DDIM 50 steps', lambda: ddim_sample(x_T, 50)),
           ('DDIM 20 steps', lambda: ddim_sample(x_T, 20)),
           ('DDIM 5 steps', lambda: ddim_sample(x_T, 5))]

fig, axes = plt.subplots(1, len(configs), figsize=(3*len(configs), 3))
for ax, (name, fn) in zip(axes, configs):
    torch.manual_seed(123)
    x_T = torch.randn(2000, 2, device=device)
    samples = fn().cpu()
    ax.scatter(samples[:,0], samples[:,1], s=2, alpha=0.4)
    ax.scatter(x_train[:500,0].cpu(), x_train[:500,1].cpu(), s=1, alpha=0.2, c='r')
    ax.set_title(name); ax.set_aspect('equal')
    ax.set_xlim(-2, 2.5); ax.set_ylim(-1.5, 2)
plt.tight_layout(); plt.show()

## 实验 2：eta 插值（确定性 vs 随机性）

In [ ]:
etas = [0.0, 0.5, 1.0]
fig, axes = plt.subplots(1, len(etas), figsize=(4*len(etas), 4))
for ax, eta in zip(axes, etas):
    torch.manual_seed(123)
    x_T = torch.randn(2000, 2, device=device)
    samples = ddim_sample(x_T, 50, eta=eta).cpu()
    ax.scatter(samples[:,0], samples[:,1], s=2, alpha=0.4)
    ax.set_title(f'DDIM eta={eta} (50 steps)'); ax.set_aspect('equal')
    ax.set_xlim(-2, 2.5); ax.set_ylim(-1.5, 2)
plt.tight_layout(); plt.show()

## 实验 3：DDIM 确定性验证

DDIM (eta=0) 应当满足：同一个 x_T 多次采样，结果完全相同。

In [ ]:
torch.manual_seed(123)
x_T_fixed = torch.randn(5, 2, device=device)
for trial in range(3):
    out = ddim_sample(x_T_fixed, 50, eta=0.0)
    print(f'trial {trial}: {out[0].cpu().numpy()}')
print('\nAll trials should be EXACTLY identical (DDIM eta=0 is deterministic)')

## 思考题

1. 把 DDIM 跳到 3 步、2 步、1 步——哪一步开始彻底失败？为什么 5 步还能勉强可看？
2. 在 DDPM 采样中固定 `torch.manual_seed(0)`，结果会一样吗？为什么 DDIM 才有真正的确定性？
3. 用同一个 x_T，DDIM 50 步和 DDPM 1000 步生成的样本"轨迹是否对应同一个目标点"？数学上预期？
4. （进阶）做一个 5x5 的网格：行=步数，列=eta，看二维质量空间